<a href="https://colab.research.google.com/github/ankitarchoudhary/IBM-Data-Science-Capstone/blob/main/Phase5_Interactive_Visual_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 5: Interactive Visual Analytics

This phase has two parts:
- **Part A: Interactive Map with Folium** - launch site locations, color-coded landing outcomes,
  and a distance calculation to the nearest coastline.
- **Part B: Interactive Dashboard with Plotly Dash** - a dropdown, pie chart, payload range
  slider, and scatter plot, all connected together in a small web app.


## Part A: Interactive Map with Folium

## Install and Import Folium

In [1]:
!pip install folium -q

**Explanation:** `folium` is not preinstalled in Colab by default, so this installs it.
The `-q` flag runs the installation quietly (less output text).

In [2]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from math import sin, cos, sqrt, atan2, radians

df = pd.read_csv('spacex_wrangled.csv')
df.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


**Explanation:**
- `folium` builds interactive maps that can be displayed directly in a notebook.
- `MarkerCluster` groups nearby markers together and expands them on zoom, which keeps the map
  readable when there are many launches at the same site.
- `math` functions (`sin`, `cos`, `sqrt`, `atan2`, `radians`) will be used later to calculate the
  real-world distance between two points on Earth.

## Prepare Launch Site Data

In [3]:
launch_sites_df = df.groupby('LaunchSite')[['Latitude', 'Longitude']].first().reset_index()
launch_sites_df

,LaunchSite,Latitude,Longitude
0,CCAFS SLC 40,28.561857,-80.577366
1,KSC LC 39A,28.608058,-80.603956
2,VAFB SLC 4E,34.632093,-120.610829


**Explanation:** `.groupby('LaunchSite')[['Latitude','Longitude']].first()` picks one
Latitude/Longitude pair per launch site (since every launch from the same site shares the same
coordinates). `.reset_index()` turns the grouped result back into a normal DataFrame with
`LaunchSite` as a regular column instead of an index.

## Markers for All Launch Sites

In [4]:
site_map = folium.Map(location=[29, -95], zoom_start=4)

for _, site in launch_sites_df.iterrows():
    coordinate = [site['Latitude'], site['Longitude']]
    folium.Circle(
        coordinate,
        radius=1000,
        color='#3186cc',
        fill=True,
        fill_color='#3186cc'
    ).add_to(site_map)
    folium.map.Marker(
        coordinate,
        icon=folium.DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html=f'<div style="font-size: 12px; color:#0b3d91;"><b>{site["LaunchSite"]}</b></div>'
        )
    ).add_to(site_map)

site_map

**Explanation:**
- `folium.Map(location=[29, -95], zoom_start=4)` creates a map centered roughly over the
  southeastern United States (where most SpaceX launch sites are), zoomed out enough to see all
  of them.
- For each launch site, `folium.Circle(...)` draws a circular marker at its coordinates.
- `folium.map.Marker(...)` with a `DivIcon` adds a small text label showing the site's name next
  to its circle, so each site is identifiable at a glance.
- Displaying `site_map` as the last line of the cell renders the interactive map directly below.

## Color-Coded Outcomes per Launch Site

In [5]:
df['marker_color'] = df['Class'].apply(lambda outcome: 'green' if outcome == 1 else 'red')

outcome_map = folium.Map(location=[29, -95], zoom_start=4)
marker_cluster = MarkerCluster().add_to(outcome_map)

for _, launch in df.iterrows():
    folium.Marker(
        location=[launch['Latitude'], launch['Longitude']],
        icon=folium.Icon(color='white', icon_color=launch['marker_color']),
        popup=f"{launch['LaunchSite']}<br>Outcome: {launch['Outcome']}"
    ).add_to(marker_cluster)

outcome_map

**Explanation:**
- `df['Class'].apply(lambda outcome: 'green' if outcome == 1 else 'red')` creates a new column
  assigning green to successful landings and red to unsuccessful ones.
- `MarkerCluster()` groups markers that are close together (which happens a lot here, since
  multiple launches share the same 3 launch site coordinates) and shows a count badge; clicking
  it expands to reveal the individual markers.
- Each marker's `popup` text shows the launch site name and outcome when clicked, and its color
  reflects whether that specific launch landed successfully.

## Distance from a Launch Site to the Nearest Coastline

In [6]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    distance = R * c
    return distance

# KSC LC-39A coordinates, and an approximate nearby coastline point (read manually from the map)
launch_site_lat, launch_site_lon = 28.608058, -80.603956
coastline_lat, coastline_lon = 28.56367, -80.57163

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f"Distance from KSC LC-39A to the nearest coastline point: {distance_coastline:.2f} km")

Distance from KSC LC-39A to the nearest coastline point: 5.86 km


**Explanation:**
- `calculate_distance` implements the Haversine formula, which calculates the great-circle
  distance between two points on Earth's surface given their latitude and longitude - a straight
  Euclidean distance would be inaccurate here since the Earth is a sphere, not a flat plane.
- `R = 6373.0` is Earth's approximate radius in kilometers.
- The coastline coordinates used here are an approximate reference point read from the map near
  KSC LC-39A; for a more precise result, you could click on the actual coastline in an interactive
  map and note the exact coordinates shown.

In [7]:
distance_map = folium.Map(location=[launch_site_lat, launch_site_lon], zoom_start=12)

folium.Marker([launch_site_lat, launch_site_lon], popup='KSC LC-39A').add_to(distance_map)
folium.Marker([coastline_lat, coastline_lon], popup='Nearest coastline point').add_to(distance_map)

folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]],
    color='blue'
).add_to(distance_map)

folium.map.Marker(
    [(launch_site_lat + coastline_lat) / 2, (launch_site_lon + coastline_lon) / 2],
    icon=folium.DivIcon(
        icon_size=(150, 20),
        icon_anchor=(0, 0),
        html=f'<div style="font-size: 12px; color:#d35400;"><b>{distance_coastline:.2f} KM</b></div>'
    )
).add_to(distance_map)

distance_map

**Explanation:**
- Two markers show the launch site and the reference coastline point.
- `folium.PolyLine(...)` draws a straight line connecting them.
- The `DivIcon` label at the midpoint of the line displays the calculated distance, so it is
  readable directly on the map.



## Part B: Interactive Dashboard with Plotly Dash

## Install Dash and Import Libraries

In [8]:
!pip install dash -q

**Explanation:** `dash` is not preinstalled in Colab, so this installs it. `plotly` (used for
the charts inside the dashboard) is preinstalled in Colab already.

In [9]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, Input, Output

df = pd.read_csv('spacex_wrangled.csv')
df.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857,0
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093,0
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857,0


**Explanation:**
- `plotly.express` provides quick, high-level chart-building functions (used here for the pie
  and scatter charts).
- `Dash` is the main class used to create the web application itself.
- `dcc` (Dash Core Components) provides interactive elements like dropdowns and sliders.
- `html` provides basic HTML building blocks (like `Div`, `H1`) to lay out the page.
- `Input` and `Output` are used to connect interactive elements to the charts that respond to them.

## Build the Dashboard Layout

In [10]:
app = Dash(__name__)
app.title = "SpaceX Launch Records Dashboard"

launch_sites = df['LaunchSite'].unique().tolist()
dropdown_options = [{'label': 'All Sites', 'value': 'ALL'}] + [
    {'label': site, 'value': site} for site in launch_sites
]

min_payload = df['PayloadMass'].min()
max_payload = df['PayloadMass'].max()

app.layout = html.Div([

    html.H1(
        "SpaceX Launch Records Dashboard",
        style={'textAlign': 'center', 'color': '#0B3D91', 'fontSize': 28}
    ),

    html.Div([
        html.Label("Select Launch Site:"),
        dcc.Dropdown(
            id='site-dropdown',
            options=dropdown_options,
            value='ALL',
            placeholder='Select a launch site',
            searchable=True
        )
    ], style={'width': '60%', 'margin': '0 auto 30px auto'}),

    html.Div([
        dcc.Graph(id='success-pie-chart')
    ]),

    html.Div([
        html.Label("Payload Mass Range (kg):"),
        dcc.RangeSlider(
            id='payload-slider',
            min=0,
            max=int(max_payload) + 1000,
            step=1000,
            value=[int(min_payload), int(max_payload)],
            marks={i: str(i) for i in range(0, int(max_payload) + 2000, 4000)}
        )
    ], style={'width': '80%', 'margin': '30px auto'}),

    html.Div([
        dcc.Graph(id='payload-scatter-chart')
    ]),
])

print("Layout defined.")

Layout defined.


**Explanation:**
- `launch_sites` and `dropdown_options` build the list of choices for the dropdown: "All Sites"
  plus each individual launch site found in the data.
- `min_payload` / `max_payload` are used to set sensible default bounds for the range slider.
- `app.layout = html.Div([...])` defines the page structure: a title, a dropdown, a pie chart
  placeholder (`dcc.Graph`), a payload range slider, and a scatter chart placeholder. The graphs
  are empty at this point - they get filled in by the callback functions defined next.
- `dcc.RangeSlider` lets the user select a minimum and maximum payload mass; `marks` adds labeled
  tick marks every 4000 kg so the slider is easy to read.

## Callback: Pie Chart of Landing Outcomes

In [11]:
@app.callback(
    Output(component_id='success-pie-chart', component_property='figure'),
    Input(component_id='site-dropdown', component_property='value')
)
def update_pie_chart(selected_site):
    if selected_site == 'ALL':
        success_by_site = df[df['Class'] == 1].groupby('LaunchSite').size().reset_index(name='count')
        fig = px.pie(
            success_by_site, values='count', names='LaunchSite',
            title='Total Successful Landings by Launch Site'
        )
    else:
        site_df = df[df['LaunchSite'] == selected_site]
        outcome_counts = site_df['Class'].value_counts().reset_index()
        outcome_counts.columns = ['Class', 'count']
        outcome_counts['Result'] = outcome_counts['Class'].map({1: 'Success', 0: 'Failure'})
        fig = px.pie(
            outcome_counts, values='count', names='Result',
            title=f'Landing Outcomes for {selected_site}'
        )
    return fig

**Explanation:**
- `@app.callback(...)` connects this function to the dashboard: whenever the dropdown's value
  changes, `update_pie_chart` runs automatically and its return value updates the pie chart.
- `Output(...)` specifies which component and property gets updated (the `figure` of the chart
  with id `success-pie-chart`); `Input(...)` specifies which component's change triggers the
  callback (the dropdown's `value`).
- When "All Sites" is selected, the chart shows how successful landings are distributed across
  the three launch sites.
- When a specific site is selected, the chart instead shows that site's own success versus
  failure split.

## Callback: Scatter Chart of Payload Mass vs. Landing Outcome

In [12]:
@app.callback(
    Output(component_id='payload-scatter-chart', component_property='figure'),
    [Input(component_id='site-dropdown', component_property='value'),
     Input(component_id='payload-slider', component_property='value')]
)
def update_scatter_chart(selected_site, payload_range):
    low, high = payload_range
    filtered_df = df[(df['PayloadMass'] >= low) & (df['PayloadMass'] <= high)]

    if selected_site != 'ALL':
        filtered_df = filtered_df[filtered_df['LaunchSite'] == selected_site]

    fig = px.scatter(
        filtered_df, x='PayloadMass', y='Class', color='LaunchSite',
        labels={'Class': 'Landing Outcome (0 = Failure, 1 = Success)'},
        title='Payload Mass vs. Landing Outcome',
        hover_data=['Orbit', 'Date']
    )
    return fig

**Explanation:**
- This callback listens to **two** inputs: the dropdown and the range slider. It runs again
  whenever either one changes.
- `payload_range` arrives as a two-item list `[low, high]` from the slider; the line
  `low, high = payload_range` unpacks it into two separate variables.
- The data is filtered first by the selected payload range, then further filtered by launch site
  if a specific one (not "All Sites") is selected.
- `px.scatter(...)` plots payload mass against the landing outcome (0 or 1), colored by launch
  site, with orbit and date shown when hovering over a point.

## Run the Dashboard

In [13]:
import threading
from google.colab import output

def run_dash_app():
    app.run(port=8050)

thread = threading.Thread(target=run_dash_app)
thread.start()

output.serve_kernel_port_as_iframe(8050, height=800)

Dash is running on http://127.0.0.1:8050/



<IPython.core.display.Javascript object>

INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'


**Explanation:**
- `app.run(...)` starts the dashboard's web server.

